In [1]:
##################################################Random Forest Grid#############################################
#Import Libraries#Import Libraries
import pandas as pd
#Read the excel file
dataset=pd.read_csv("CKD.csv")
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset


,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2.000000,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,0,0,0,0,0,0,1,1,0,1
1,3.000000,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,1,0,0,0,0,0,1,0,0,1
2,4.000000,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,1,0,0,0,0,0,1,0,0,1
3,5.000000,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,1,0,0,0,0,0,1,0,1,1
4,5.000000,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,1,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,0.0,0.0,219.000000,36.000000,1.300000,139.000000,3.700000,12.500000,...,1,0,0,0,0,0,1,0,0,1
395,51.492308,70.000000,0.0,2.0,220.000000,68.000000,2.800000,137.528754,4.627244,8.700000,...,1,0,0,1,1,0,1,0,1,1
396,51.492308,70.000000,3.0,0.0,110.000000,115.000000,6.000000,134.000000,2.700000,9.100000,...,1,0,0,1,1,0,0,0,0,1
397,51.492308,90.000000,0.0,0.0,207.000000,80.000000,6.800000,142.000000,5.500000,8.500000,...,1,0,0,1,1,0,1,0,1,1


In [2]:
dataset["classification_yes"].value_counts()

classification_yes
1    249
0    150
Name: count, dtype: int64

In [5]:
#displaying columns
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [7]:
#Intependent variable Or input variable
independent=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
#dependent variable  OR Output Variable
dependent=dataset[["classification_yes"]]
independent.shape

(399, 27)

In [9]:
dependent

,classification_yes
0,1
1,1
2,1
3,1
4,1
...,...
394,1
395,1
396,1
397,1


In [11]:
#Train and Test(Split)
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(independent,dependent,test_size=0.30,random_state=0)

In [12]:
#####################standardization######################
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train_ = sc.fit_transform(x_train)
x_test_ = sc.transform(x_test)

In [17]:
from sklearn.neighbors import KNeighborsClassifier

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
               'n_neighbors': [3, 5, 7, 9],
               'weights': ['uniform', 'distance'],
              'metric': ['euclidean', 'manhattan']} 



grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1') 
   
# fitting the model for grid search 
grid.fit(x_train_, y_train) 

Fitting 5 folds for each of 16 candidates, totalling 80 fits


C:\Users\Chezhian\anaconda3\Lib\site-packages\sklearn\neighbors\_classification.py:215: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


GridSearchCV(estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': [3, 5, 7, 9],
                         'weights': ['uniform', 'distance']},
             scoring='f1', verbose=3)

In [20]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(x_test_) 
   


In [21]:
####Confusion Matrix#####################
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
print("Confusion Matrix:\n",cm)

Confusion Matrix:
 [[45  0]
 [ 6 69]]


In [25]:
# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)
print("classification report:\n",clf_report)

classification report:
               precision    recall  f1-score   support

           0       0.88      1.00      0.94        45
           1       1.00      0.92      0.96        75

    accuracy                           0.95       120
   macro avg       0.94      0.96      0.95       120
weighted avg       0.96      0.95      0.95       120



In [27]:
#############ROC Auc-Receiver Operating Characteristic Curve(ROc=1 perfect model;ROC=0.9 to .99 Excelent model;
############################################################ROC=0.8 to .89 Good;ROC=0.7 to .70 Fair;ROC=0.6 to .69 Poor;<=5 Random guess
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test_)[:,1])

0.9995555555555555